# QLoRA Fine-tune â€” Text-to-SQL on Spider (Colab T4)

Phase 1 of the Multi-Agent Analyst System. Fine-tunes a 4-bit base model with QLoRA so it reliably turns *(schema + question)* into SQL. The exported adapter is loaded later by `src/tools/text_to_sql.py` as a live agent tool.

**Defaults (per spec):** 4-bit base, LoRA `r=16`, `alpha=32`, target *all* linear layers, LR `2e-4`, gradient checkpointing. Fits on a **free Colab T4**.

**Runtime:** set `Runtime > Change runtime type > T4 GPU` before running.

In [4]:
# 1. Install â€” LOCAL GPU (RTX A4500). Assumes a working CUDA + PyTorch already.
#%%capture
!pip install unsloth
!pip install --upgrade datasets trl peft accelerate bitsandbytes
# On Colab instead use:
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" datasets

  Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached numpy-2.4.6-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached pandas-3.0.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached xxhash-3.7.0-cp312-cp312-win_amd64.whl.metadata (13 kB)
  Using cached aiohttp-3.14.1-cp312-cp312-win_amd64.whl.metadata (8.5 kB)
  Using cached anyio-4.14.0-py3-none-any

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.6.7 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth 2026.6.7 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.6.0 which is incompatible.
unsloth-zoo 2026.6.5 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth-zoo 2026.6.5 requires trl!=0.19.0,<=0.24.0,>=0.18.2; sys_platform != "darwin" or platform_machine != "arm64", but you have trl 1.6.0 which is incompatible.


In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"   # avoid hf-xet "fast download" stalls on Windows

# 2. Config â€” tuned for a single NVIDIA RTX A4500 (20 GB, Ampere / bf16)
BASE_MODEL    = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # 4-bit base
MAX_SEQ_LEN   = 1024          # schema+question+SQL is short; 1024 is plenty & much faster than 2048
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.0
LEARNING_RATE = 2e-4
EPOCHS        = 1
BATCH_SIZE    = 2             # was 8 -> maxed 20GB VRAM, forcing slow CPU gradient-offload thrash.
                              # 2 fits with headroom so unsloth does NOT offload -> far faster/step.
GRAD_ACCUM    = 8             # effective batch = 16 (unchanged: 2 x 8), so training result is same
MAX_STEPS     = 600           # TIME CAP â€” enough to show a clear before/after delta (~10-15 min).
                              # Set to 0 to train a full epoch instead (slower, marginally better).
MAX_TRAIN     = 0             # 0 = all rows; MAX_STEPS already bounds wall-clock
OUTPUT_DIR    = "outputs"
ADAPTER_DIR   = "adapter"
HF_REPO       = "abhiram3000/llama31-sql-qlora" 

## Data â€” build Spider instruction dataset

Either upload `finetune/data/train.jsonl` (produced by `prepare_spider.py`) to the Colab session, or run the prep inline below. The prompt template **must match** `prepare_spider.py` and `text_to_sql.py` exactly.

In [6]:
# 3. Build the dataset (schema-inline, Spider-derived â€” loads cleanly, no script)
from datasets import load_dataset

SYSTEM = (
    "You are a precise text-to-SQL engine. Given a database schema and a question, "
    "output a single valid SQLite query that answers it. Output ONLY the SQL, no prose."
)
PROMPT_TEMPLATE = (
    "{system}\n\n### Database schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"
)

# Each row: question, context (CREATE TABLE schema), answer (SQL). Spider + WikiSQL
# derived; schema is inline so no fragile tables.json assembly. (The canonical
# `spider` loader is script-based and breaks on newer `datasets`.)
ds = load_dataset("b-mc2/sql-create-context", split="train")

EOS = "<|eot_id|>"  # Llama-3.1 end token
def to_text(ex):
    prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=ex["context"], question=ex["question"])
    return {"text": prompt + ex["answer"].strip() + EOS}

train_ds = ds.map(to_text, remove_columns=ds.column_names)
if MAX_TRAIN:
    train_ds = train_ds.select(range(min(MAX_TRAIN, len(train_ds))))
print(train_ds)
print(train_ds[0]["text"][:600])

Dataset({
    features: ['text'],
    num_rows: 78577
})
You are a precise text-to-SQL engine. Given a database schema and a question, output a single valid SQLite query that answers it. Output ONLY the SQL, no prose.

### Database schema:
CREATE TABLE head (age INTEGER)

### Question:
How many heads of the departments are older than 56 ?

### SQL:
SELECT COUNT(*) FROM head WHERE age > 56<|eot_id|>


In [3]:
# 4. Load 4-bit base + attach LoRA (all linear layers)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],  # all linear
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

ðŸ¦¥ Unsloth: Will patch your computer to enable 2x faster free finetuning.


d:\Anirudh\Abhiram\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0618 16:57:54.769000 23196 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


ðŸ¦¥ Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A4500. Num GPUs = 1. Max memory: 19.99 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 291/291 [00:02<00:00, 117.49it/s]
Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [7]:
# 5. Train (QLoRA via TRL SFTTrainer) â€” optimized for RTX A4500
# Speedups vs the T4 defaults: bf16 (Ampere), packing=True (no padding waste),
# bigger batch, shorter seq len, and a max_steps time cap.
# IMPORTANT: pass tokenizer=tokenizer directly â€” do NOT rename it to processing_class.
# NOTE: dataset_num_proc=None â€” REQUIRED on Windows. In datasets>=4.x, num_proc=1 still
# uses a Pool(1) worker that dies in Jupyter ("subprocess abruptly died"); only None
# takes the pure single-process path. Unsloth keeps None as-is on spawn (Windows).
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

targs = dict(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS else -1,   # -1 => train by epochs
    learning_rate=LEARNING_RATE,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),               # A4500 -> bf16 = True
    logging_steps=20,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir=OUTPUT_DIR,
    report_to="none",
)

# packing=True concatenates short examples to fill MAX_SEQ_LEN -> far fewer steps,
# big throughput win for short SQL samples.
try:
    from trl import SFTConfig
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
                       packing=True, dataset_num_proc=None, **targs),
    )
except (ImportError, TypeError):
    from transformers import TrainingArguments
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=True,
        dataset_num_proc=None,
        args=TrainingArguments(**targs),
    )

trainer.train()

ðŸ¦¥ Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,132 | Num Epochs = 2 | Total steps = 600
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,1.541580
40,0.665388
60,0.619010
80,0.588008
100,0.583895
120,0.562774
140,0.551877
160,0.555745
180,0.543132
200,0.533431


Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-500\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-600\tokenizer_config.json.


TrainOutput(global_step=600, training_loss=0.5599176025390625, metrics={'train_runtime': 8178.316, 'train_samples_per_second': 1.174, 'train_steps_per_second': 0.073, 'total_flos': 4.29134157710721e+17, 'train_loss': 0.5599176025390625, 'epoch': 1.1790457452041319})

In [9]:
import os
# Set your HF token via the environment (do NOT hardcode/commit it):
#   PowerShell:  $env:HF_TOKEN="hf_xxx"      bash:  export HF_TOKEN=hf_xxx
#   or just run:  huggingface-cli login
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # avoid hf-xet stalls on Windows
HF_REPO = "abhiram3000/llama31-sql-qlora"   # adapter target on the Hub


In [10]:
# 6. Save the LoRA adapter (small â€” just the adapter, not the base)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)

# Optional: push to HF Hub so the app / Spaces can pull it at runtime.
# Locally, either set HF_TOKEN in the environment or run `huggingface-cli login` first.
if HF_REPO:
    import os
    from huggingface_hub import login
    token = os.environ.get("HF_TOKEN")
    login(token) if token else login()   # falls back to cached credentials
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print("Pushed to", HF_REPO)

# Then point the app at it: set SQL_ADAPTER_REPO=<HF_REPO> in .env, OR copy the
# adapter/ folder into the repo at finetune/adapter/.

d:\Anirudh\Abhiram\.venv\Lib\site-packages\peft\utils\other.py:1419: UserWarning: Unable to fetch remote file due to the following error The read operation timed out - silently ignoring the lookup for the file config.json in unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit.
  warnings.warn(
d:\Anirudh\Abhiram\.venv\Lib\site-packages\peft\utils\save_and_load.py:372: UserWarning: Could not find a config file in unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit - will assume that the vocabulary was not modified.
  warnings.warn(


Saved adapter to adapter


adapter_model.safetensors: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 168M/168M [00:33<00:00, 4.96MB/s] 


Saved model to https://huggingface.co/abhiram3000/llama31-sql-qlora


Unsloth: Restored added_tokens_decoder metadata in C:\Users\HP\AppData\Local\Temp\tmpbi_s90n1\tokenizer_config.json.
tokenizer.json: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 17.2M/17.2M [00:03<00:00, 4.56MB/s]


Pushed to abhiram3000/llama31-sql-qlora


In [11]:
# 7. Quick sanity inference
FastLanguageModel.for_inference(model)
schema = "CREATE TABLE singer (name TEXT, country TEXT, age INT);"
q = "What are the names of singers from France, oldest first?"
prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=schema, question=q)
ids = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**ids, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
d:\Anirudh\Abhiram\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
d:\Anirudh\Abhiram\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


SELECT name FROM singer WHERE country = "France" ORDER BY age DESC


## Done â€” what to copy off the GPU box

After all cells run you have, on this machine:
1. **`eval_report.json`** + the printed `BEFORE â†’ AFTER` execution-accuracy numbers â€” paste them into the repo's `README.md` proof-point table and `finetune/README.md`.
2. **The adapter on HF Hub** (if you set `HF_REPO` and were logged in) â€” note the repo id, e.g. `your-username/llama31-sql-qlora`.

Then on your normal machine (no GPU needed): set `SQL_ADAPTER_REPO=<that repo id>` in `.env`, update the README numbers, and commit. (The adapter only *runs* on a GPU â€” locally the app keeps using the Groq fallback, which is expected and documented.)

In [14]:
# 8. Load Spider dev set + real databases (for execution accuracy)
# NOTE: the HF `xlangai/spider` repo now ships PARQUET only (no dev.json / tables.json /
# database/ sqlite files), so snapshot_download can't give us runnable DBs. Instead we use
# the official Spider zip (file id 1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J from
# https://yale-lily.github.io/spider), already downloaded & unzipped to spider_extracted/.
# To re-fetch from scratch:  pip install gdown
#   python -m gdown 1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J -O spider_data.zip
#   python -c "import zipfile; zipfile.ZipFile('spider_data.zip').extractall('spider_extracted')"
import glob, json, os

SPIDER_ROOT = "spider_extracted/spider_data"   # contains dev.json, tables.json, database/
dev_json    = os.path.join(SPIDER_ROOT, "dev.json")
tables_json = os.path.join(SPIDER_ROOT, "tables.json")
DB_DIR      = os.path.join(SPIDER_ROOT, "database")
print("dev.json:", dev_json, "\ntables.json:", tables_json, "\ndatabase/:", DB_DIR)
assert os.path.exists(dev_json) and os.path.exists(tables_json) and os.path.isdir(DB_DIR), (
    "Spider files not found under spider_extracted/. Re-download the official zip: "
    "`python -m gdown 1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J -O spider_data.zip` then "
    "`python -c \"import zipfile; zipfile.ZipFile('spider_data.zip').extractall('spider_extracted')\"`"
)

dev = json.load(open(dev_json))
tables = {t["db_id"]: t for t in json.load(open(tables_json))}

def schema_for(db_id):
    m = tables[db_id]
    names, cols, types = m["table_names_original"], m["column_names_original"], m["column_types"]
    per = {i: [] for i in range(len(names))}
    for ci, (ti, cn) in enumerate(cols):
        if ti != -1:
            per[ti].append(f"{cn} {types[ci].upper()}")
    return "\n".join(f"CREATE TABLE {names[ti]} ({', '.join(per[ti])});" for ti in range(len(names)))

print("dev examples:", len(dev))

dev.json: spider_extracted/spider_data\dev.json 
tables.json: spider_extracted/spider_data\tables.json 
database/: spider_extracted/spider_data\database
dev examples: 1034


In [15]:
# 9. Execution accuracy: BASE vs FINE-TUNED on real Spider dev DBs (runs on this GPU)
import json, os, sqlite3, torch
from unsloth import FastLanguageModel

LIMIT = 200   # held-out dev questions to score (raise for a tighter estimate)

# A separate frozen base model for the "before" number (unambiguous vs the adapter).
# Two 4-bit 8B models fit in 20 GB.
base_model_eval, base_tok = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
FastLanguageModel.for_inference(base_model_eval)
FastLanguageModel.for_inference(model)

def _clean(t):
    t = t.replace("```sql", "").replace("```", "").strip()
    for s in ("\n###", "\nQuestion:", "\n--", "\n\n"):
        if s in t:
            t = t.split(s)[0]
    t = t.strip()
    return (t.split(";")[0] + ";").strip() if ";" in t else t

def _gen(m, tok, prompt):
    ids = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = m.generate(**ids, max_new_tokens=128, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return _clean(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

def _run(db, sql):
    try:
        con = sqlite3.connect(f"file:{db}?mode=ro", uri=True)
        con.text_factory = lambda b: b.decode("utf-8", "ignore")
        rows = con.execute(sql).fetchall(); con.close()
        return True, sorted(map(repr, rows))
    except Exception:
        return False, None

def evaluate(m, tok):
    correct = runnable = total = 0
    for ex in dev[:LIMIT]:
        db = os.path.join(DB_DIR, ex["db_id"], ex["db_id"] + ".sqlite")
        if not os.path.exists(db):
            continue
        total += 1
        prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=schema_for(ex["db_id"]),
                                        question=ex["question"])
        gok, g = _run(db, ex["query"])
        pok, p = _run(db, _gen(m, tok, prompt))
        runnable += int(pok)
        correct += int(gok and p == g)
    return dict(total=total,
                exec_accuracy=round(correct / total, 4) if total else 0.0,
                runnable_rate=round(runnable / total, 4) if total else 0.0)

before = evaluate(base_model_eval, base_tok); print("BEFORE (base):     ", before)
after  = evaluate(model, tokenizer);          print("AFTER  (fine-tuned):", after)

report = dict(base_model=BASE_MODEL, n=before["total"], before=before, after=after,
              exec_accuracy_delta=round(after["exec_accuracy"] - before["exec_accuracy"], 4))
open("eval_report.json", "w").write(json.dumps(report, indent=2))
print(f"\nEXEC-ACCURACY  {before['exec_accuracy']:.1%} -> {after['exec_accuracy']:.1%}  "
      f"(delta {report['exec_accuracy_delta']:+.1%})")
print("Saved eval_report.json. Copy these numbers into README.md and finetune/README.md.")

==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A4500. Num GPUs = 1. Max memory: 19.99 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 291/291 [00:02<00:00, 107.94it/s]
Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have

BEFORE (base):      {'total': 200, 'exec_accuracy': 0.49, 'runnable_rate': 0.775}


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

AFTER  (fine-tuned): {'total': 200, 'exec_accuracy': 0.185, 'runnable_rate': 0.685}

EXEC-ACCURACY  49.0% -> 18.5%  (delta -30.5%)
Saved eval_report.json. Copy these numbers into README.md and finetune/README.md.


In [16]:
# 9b. DIAGNOSTIC: eyeball where the fine-tuned model diverges from base/gold.
# Runs in the same kernel (models already loaded). Shows the failure mode directly.
import textwrap

for ex in dev[:8]:
    db = os.path.join(DB_DIR, ex["db_id"], ex["db_id"] + ".sqlite")
    prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=schema_for(ex["db_id"]),
                                    question=ex["question"])
    base_sql = _gen(base_model_eval, base_tok, prompt)
    ft_sql   = _gen(model, tokenizer, prompt)
    gok, g = _run(db, ex["query"])
    bok, b = _run(db, base_sql)
    fok, f = _run(db, ft_sql)
    print("Q:   ", ex["question"])
    print("GOLD:", ex["query"])
    print(f"BASE [{'run ' if bok else 'ERR '}{'âœ“' if (bok and b==g) else 'âœ—'}]:", base_sql)
    print(f"FT   [{'run ' if fok else 'ERR '}{'âœ“' if (fok and f==g) else 'âœ—'}]:", ft_sql)
    print("-" * 100)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q:    How many singers do we have?
GOLD: SELECT count(*) FROM singer
BASE [run âœ“]: SELECT COUNT(*) FROM singer;
FT   [run âœ—]: SELECT COUNT(DISTINCT Singer_ID) FROM singer_in_concert
----------------------------------------------------------------------------------------------------


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q:    What is the total number of singers?
GOLD: SELECT count(*) FROM singer
BASE [run âœ“]: SELECT COUNT(Singer_ID) FROM singer;
FT   [run âœ—]: SELECT COUNT(*) FROM singer_in_concert AS T1 JOIN singer AS T2 ON T1.Singer_ID = T2.Singer_ID JOIN concert AS T3 ON T1.concert_ID = T3.concert_ID WHERE T2.Name = "Taylor Swift" AND T3.Theme = "Rock" AND T3.Year = 2010
----------------------------------------------------------------------------------------------------


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q:    What is the average, minimum, and maximum age for all French singers?
GOLD: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
BASE [ERR âœ—]: SELECT AVG(Age), MIN(Age), MAX(Age) FROM singer WHERE Country = 'France'  ### SQL: 
SELECT AVG(Age), MIN(Age), MAX(Age) FROM singer WHERE Country = 'France'  
SELECT AVG(Age), MIN(Age), MAX(Age) FROM singer WHERE Country = 'France'
 
SELECT AVG(Age), MIN(Age), MAX(Age) FROM singer WHERE Country = 'France'
 
SELECT AVG(Age), MIN(Age), MAX(Age) FROM singer WHERE Country = 'France'
 
SELECT AVG
FT   [run âœ“]: SELECT AVG(Age), MIN(Age), MAX(Age) FROM singer WHERE Country = "France"
----------------------------------------------------------------------------------------------------


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q:    Show the name and the release year of the song by the youngest singer.
GOLD: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
BASE [run âœ—]: SELECT T2.Name, T2.Song_release_year FROM singer AS T1 JOIN singer AS T2 ON T1.Age = T2.Age WHERE T1.Age = ( SELECT MIN(Age) FROM singer );
FT   [ERR âœ—]: SELECT T1.Name, T1.Song_Name, T1.Song_release_year FROM singer AS T1 JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_ID JOIN concert AS T3 ON T2.concert_ID = T3.concert_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer) AND T3.Theme = "Rock" AND T3.Year = 2010 AND T1.Is_male = "Male" AND T1.Country = "USA" AND T3.Stadium_ID IN (SELECT Stadium_ID FROM stadium WHERE Name = "Stadium Name" AND
----------------------------------------------------------------------------------------------------


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q:    What are the names and release years for all the songs of the youngest singer?
GOLD: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
BASE [run âœ“]: SELECT T2.Song_Name, T2.Song_release_year FROM singer AS T1 JOIN singer AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age = ( SELECT MIN(T1.Age) FROM singer AS T1 ) AND T1.Singer_ID = T2.Singer_ID;
FT   [ERR âœ—]: SELECT T1.Song_Name, T1.Song_release_year FROM singer AS T1 JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_ID JOIN concert AS T3 ON T2.concert_ID = T3.concert_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer) AND T3.Theme = "Rock" AND T3.Year = 2010 AND T1.Is_male = "Male" AND T1.Country = "USA" AND T3.Stadium_ID IN (SELECT Stadium_ID FROM stadium WHERE Name = "Stadium Name" AND Capacity > 100
----------------------------------------------------------------------------------------------------


## Next: measure execution accuracy (before vs after)

Download Spider's `database/` folder, then run `finetune/evaluate_sql.py` to get the **before/after exec-accuracy** numbers for the README proof-point table.

```bash
python finetune/evaluate_sql.py --spider-db-dir spider/database --adapter adapter --limit 200
```